In [20]:
import pandas as pd

# data = pd.read_csv('rulesm_raw/rulesm_part_2.csv')
data = pd.read_csv('rulesm_test - english_dataset.csv')

In [21]:
data

,url,paragraph_1,paragraph_2,span_1,span_2,Label,Anchor_span,subset,Unnamed: 8,Unnamed: 9
0,https://www.govinfo.gov/app/collection/uscode/...,Committee on Education and the Workforce of Ho...,Committee on Education and the Workforce of Ho...,NaN,Committee on Education and Labor of House of R...,Addition,Committee on Education and the Workforce of Ho...,"TITLE 2, CHAPTER 2 - ORGANIZATION OF CONGRESS",NaN,NaN
1,https://www.govinfo.gov/app/collection/uscode/...,‘‘(1) the Committee on District of Columbia of...,‘‘(1) the Committee on District of Columbia of...,Reform,Accountability,Contradiction,NaN,"TITLE 2, CHAPTER 2 - ORGANIZATION OF CONGRESS",NaN,NaN
2,https://www.govinfo.gov/app/collection/uscode/...,Committee on Oversight and Government Reform o...,Committee on Oversight and Government Reform o...,NaN,Committee on Oversight and Reform of House of ...,Addition,Committee on Oversight and Government Reform o...,"TITLE 2, CHAPTER 2 - ORGANIZATION OF CONGRESS",NaN,NaN
3,NaN,F. Those functions of the Office of Management...,F. Those functions of the Office of Management...,section 7 of the Federal Advisory Committee Ac...,5 U.S.C. 1006,Contradiction,NaN,"TITLE 3, CHAPTER 2",NaN,2020-2024
4,NaN,"Ex. Ord. No. 13283, Jan. 21, 2003, 68 F.R. 337...","Ex. Ord. No. 13283, Jan. 21, 2003, 68 F.R. 337...",NaN,formerly,Addition,set out as a note,"TITLE 3, CHAPTER 3",NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
74,NaN,When applicable this table shall indicate sepa...,"When applicable, that table shall also indicat...",NaN,also,Addition,table shall,NaN,NaN,NaN
75,NaN,When applicable this table shall indicate sepa...,"When applicable, that table shall also indicat...",in application of,pursuant to,Equivalent,NaN,NaN,NaN,NaN
76,NaN,The scheme of authorisations for vine planting...,The scheme of authorisations for vine planting...,to 31 December 2045,NaN,Addition,NaN,NaN,NaN,NaN
77,NaN,The scheme of authorisations for vine planting...,The scheme of authorisations for vine planting...,two mid-term reviews,a review,Contradiction,NaN,NaN,NaN,NaN


In [22]:
import pandas as pd

data['span_1'] = data['span_1'].fillna("")
data['span_2'] = data['span_2'].fillna("")

group_cols = ["paragraph_1", "paragraph_2"]
list_cols  = ["span_1", "span_2", "Label", "Anchor_span"]

data = data[data['Label'].isin(['Contradiction', 'Addition'])]
# data['Anchor_span'] = data['итоговый anchor'].fillna("")

result = (
    data.groupby(group_cols, as_index=False)
      .agg({col: list for col in list_cols})
)

In [23]:
result

,paragraph_1,paragraph_2,span_1,span_2,Label,Anchor_span
0,(A) individual or individual and spouse engage...,(A) individual or individual and spouse engage...,"[3,237,000]","[10,000,000]",[Contradiction],[nan]
1,"(A) subject to subparagraph (B), means a perso...","(A) subject to subparagraph (B), means a perso...",[or operating real property or activities inci...,"[single asset real estate, , less than 50 perc...","[Contradiction, Addition, Contradiction]","[nan, that has aggregate noncontingent liquida..."
2,(B) does not include any member of a group of ...,(B) does not include any member of a group of ...,[],[under this title],[Addition],[debtors]
3,"(B) who is appointed to an advisory committee,...","(B) who is appointed to an advisory committee,...","[3(2), the Federal Advisory Committee Act]","[1001, title 5]","[Contradiction, Contradiction]","[nan, nan]"
4,An amount of EUR 11 327 310 073 in current pri...,An amount of EUR 14 227 310 073 in current pri...,[11 327],[14 227],[Contradiction],[nan]
5,An operator that is a natural person or a micr...,An operator that is a natural person or a micr...,[],[downstream],[Addition],[operator]
6,By making available the due diligence statemen...,By making the due diligence statement availabl...,[],"[or, in the case of micro or small primary ope...",[Addition],[By making the due diligence statement availab...
7,By the authority vested in me as President by ...,By the authority vested in me as President by ...,[to assist faith-based and other organizations...,[in order to better serve people in need throu...,"[Contradiction, Contradiction]","[nan, nan]"
8,By way of derogation from the first subparagra...,By way of derogation from the first subparagra...,"[and, if duly justified,, ]","[, and commitments to convert to or maintain o...","[Addition, Addition]",[commitments for agricultural practices benefi...
9,Committee on Education and the Workforce of Ho...,Committee on Education and the Workforce of Ho...,[],[Committee on Education and Labor of House of ...,[Addition],[Committee on Education and the Workforce of H...


In [24]:
pd.options.display.max_colwidth = 50

In [25]:
import re
from difflib import SequenceMatcher

TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)

def tokenize_with_offsets(text: str):
    # returns list of (tok, start, end)
    return [(m.group(0), m.start(), m.end()) for m in TOKEN_RE.finditer(text)]

PRESERVE = set(list("()[]{}«»\"'„“”–—-"))

def norm_tok(tok: str) -> str:
    t = tok.strip()
    if t in PRESERVE:
        return t
    return re.sub(r"\W+", "", t.lower(), flags=re.UNICODE)
def extend_right_closing_punct(text: str, e: int) -> int:
    # съедаем пробелы + закрывающую пунктуацию, но стопаемся перед буквой/цифрой
    while e < len(text):
        ch = text[e]
        if ch.isspace():
            e += 1
            continue
        if ch in ")]}»\"'”":
            e += 1
            continue
        break
    return e



def find_all_occurrences(text: str, span: str):
    # более устойчиво к пробелам: " " -> "\s+"
    # если надо игнорировать пунктуацию — лучше искать по токенам (см. ниже)
    if not span or not span.strip():
        return []
    pat = re.escape(span.strip())
    pat = re.sub(r"\\\s+", r"\\s+", pat)  # пробелы в span -> \s+
    return [(m.start(), m.end()) for m in re.finditer(pat, text, flags=re.UNICODE)]

def merge_intervals(intervals):
    intervals = sorted(intervals)
    merged = []
    for s, e in intervals:
        if not merged or s > merged[-1][1]:
            merged.append([s, e])
        else:
            merged[-1][1] = max(merged[-1][1], e)
    return [(a, b) for a, b in merged]

def subtract_intervals(interval, holes):
    # interval: (s,e), holes: list of (hs,he) merged, return list of остатки
    s, e = interval
    out = []
    cur = s
    for hs, he in holes:
        if he <= cur:
            continue
        if hs >= e:
            break
        if hs > cur:
            out.append((cur, min(hs, e)))
        cur = max(cur, he)
        if cur >= e:
            break
    if cur < e:
        out.append((cur, e))
    return out

def protected_ranges(text, spans):
    # пытаемся найти все точные/почти точные вхождения и объединить
    occ = []
    for sp in spans:
        occ.extend(find_all_occurrences(text, sp))
    return merge_intervals(occ)

def extract_equivalents(par1, par2, spans1, spans2, min_chars=30):
    t1 = tokenize_with_offsets(par1)
    t2 = tokenize_with_offsets(par2)

    n1 = [norm_tok(tok) for tok,_,_ in t1]
    n2 = [norm_tok(tok) for tok,_,_ in t2]

    # убираем пустые норм-токены (пунктуацию), но сохраняем маппинг индексов
    idx1 = [i for i, x in enumerate(n1) if x]
    idx2 = [j for j, x in enumerate(n2) if x]
    n1c = [n1[i] for i in idx1]
    n2c = [n2[j] for j in idx2]

    sm = SequenceMatcher(None, n1c, n2c, autojunk=False)

    prot1 = protected_ranges(par1, spans1)
    prot2 = protected_ranges(par2, spans2)

    eq1, eq2 = [], []
    for i1, j1, size in sm.get_matching_blocks():
        if size == 0:
            continue

        # границы блока в "очищенных" токенах -> в исходных токенах -> в char offsets
        a1 = idx1[i1]
        b1 = idx1[i1 + size - 1]
        a2 = idx2[j1]
        b2 = idx2[j1 + size - 1]

        s1 = t1[a1][1]
        e1 = t1[b1][2]
        s2 = t2[a2][1]
        e2 = t2[b2][2]

        e1 = extend_right_closing_punct(par1, e1)
        e2 = extend_right_closing_punct(par2, e2)

        # вычесть protected
        parts1 = subtract_intervals((s1, e1), prot1)
        parts2 = subtract_intervals((s2, e2), prot2)

        # обычно parts1 и parts2 совпадают по количеству, но не всегда.
        # берем попарно по порядку (можно усложнить матчингом по длине)
        k = min(len(parts1), len(parts2))
        for p in range(k):
            ss1, ee1 = parts1[p]
            ss2, ee2 = parts2[p]
            if (ee1 - ss1) >= min_chars and (ee2 - ss2) >= min_chars:
                eq1.append((ss1, ee1, par1[ss1:ee1]))
                eq2.append((ss2, ee2, par2[ss2:ee2]))

    return eq1, eq2


In [26]:
import re
import ast
import pandas as pd

# --- helpers ---

def ensure_list(x):
    """span columns sometimes are already lists, sometimes strings like "['a','b']"."""
    if isinstance(x, list):
        return x
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        # try parse python-list-like string
        if s.startswith("[") and s.endswith("]"):
            try:
                v = ast.literal_eval(s)
                return v if isinstance(v, list) else [v]
            except Exception:
                return [x]
        return [x]
    # fallback
    try:
        return list(x)
    except Exception:
        return [x]

_ws = re.compile(r"\s+", flags=re.UNICODE)

def norm_key(a: str, b: str) -> str:
    a_n = _ws.sub(" ", (a or "").strip().lower())
    b_n = _ws.sub(" ", (b or "").strip().lower())
    return a_n + "||" + b_n


def add_equivalents_row(row, *, min_chars=30):
    spans1 = ensure_list(row["span_1"])
    spans2 = ensure_list(row["span_2"])
    labels = ensure_list(row["Label"])

    eq1, eq2 = extract_equivalents(
        row["paragraph_1"],
        row["paragraph_2"],
        spans1,
        spans2,
        min_chars=min_chars,
    )

    # extract texts
    cand_pairs = [(t1, t2) for (_, _, t1), (_, _, t2) in zip(eq1, eq2)]

    # dedupe (pairwise) + drop empty/too-short after strip
    seen = set()
    pairs = []
    for a, b in cand_pairs:
        if not a or not b:
            continue
        a_s, b_s = a.strip(), b.strip()
        if len(a_s) < min_chars or len(b_s) < min_chars:
            continue
        k = norm_key(a_s, b_s)
        if k in seen:
            continue
        seen.add(k)
        pairs.append((a_s, b_s))

    # extend
    span_1_ext = spans1 + [a for a, _ in pairs]
    span_2_ext = spans2 + [b for _, b in pairs]
    label_ext  = labels + ["Equivalent"] * len(pairs)

    return pd.Series(
        {
            "span_1_ext": span_1_ext,
            "span_2_ext": span_2_ext,
            "Label_ext": label_ext,
            "n_equivalent_added": len(pairs),
        }
    )


# --- run on whole dataframe ---
out = result.apply(add_equivalents_row, axis=1)

result = pd.concat([result, out], axis=1)

# quick sanity checks
print(result["n_equivalent_added"].describe())
print(result["n_equivalent_added"].value_counts().head(20))


count    44.000000
mean      1.772727
std       0.803007
min       0.000000
25%       1.000000
50%       2.000000
75%       2.000000
max       4.000000
Name: n_equivalent_added, dtype: float64
n_equivalent_added
2    20
1    16
3     6
4     1
0     1
Name: count, dtype: int64


In [27]:
result

,paragraph_1,paragraph_2,span_1,span_2,Label,Anchor_span,span_1_ext,span_2_ext,Label_ext,n_equivalent_added
0,(A) individual or individual and spouse engage...,(A) individual or individual and spouse engage...,"[3,237,000]","[10,000,000]",[Contradiction],[nan],"[3,237,000, (A) individual or individual and s...","[10,000,000, (A) individual or individual and ...","[Contradiction, Equivalent, Equivalent]",2
1,"(A) subject to subparagraph (B), means a perso...","(A) subject to subparagraph (B), means a perso...",[or operating real property or activities inci...,"[single asset real estate, , less than 50 perc...","[Contradiction, Addition, Contradiction]","[nan, that has aggregate noncontingent liquida...",[or operating real property or activities inci...,"[single asset real estate, , less than 50 perc...","[Contradiction, Addition, Contradiction, Equiv...",2
2,(B) does not include any member of a group of ...,(B) does not include any member of a group of ...,[],[under this title],[Addition],[debtors],"[, (B) does not include any member of a group ...","[under this title, (B) does not include any me...","[Addition, Equivalent, Equivalent]",2
3,"(B) who is appointed to an advisory committee,...","(B) who is appointed to an advisory committee,...","[3(2), the Federal Advisory Committee Act]","[1001, title 5]","[Contradiction, Contradiction]","[nan, nan]","[3(2), the Federal Advisory Committee Act, (B)...","[1001, title 5, (B) who is appointed to an adv...","[Contradiction, Contradiction, Equivalent]",1
4,An amount of EUR 11 327 310 073 in current pri...,An amount of EUR 14 227 310 073 in current pri...,[11 327],[14 227],[Contradiction],[nan],"[11 327, 310 073 in current prices of the amou...","[14 227, 310 073 in current prices of the amou...","[Contradiction, Equivalent, Equivalent]",2
5,An operator that is a natural person or a micr...,An operator that is a natural person or a micr...,[],[downstream],[Addition],[operator],"[, An operator that is a natural person or a m...","[downstream, An operator that is a natural per...","[Addition, Equivalent, Equivalent]",2
6,By making available the due diligence statemen...,By making the due diligence statement availabl...,[],"[or, in the case of micro or small primary ope...",[Addition],[By making the due diligence statement availab...,"[, the operator shall assume responsibility fo...","[or, in the case of micro or small primary ope...","[Addition, Equivalent]",1
7,By the authority vested in me as President by ...,By the authority vested in me as President by ...,[to assist faith-based and other organizations...,[in order to better serve people in need throu...,"[Contradiction, Contradiction]","[nan, nan]",[to assist faith-based and other organizations...,[in order to better serve people in need throu...,"[Contradiction, Contradiction, Equivalent]",1
8,By way of derogation from the first subparagra...,By way of derogation from the first subparagra...,"[and, if duly justified,, ]","[, and commitments to convert to or maintain o...","[Addition, Addition]",[commitments for agricultural practices benefi...,"[and, if duly justified,, , By way of derogati...","[, and commitments to convert to or maintain o...","[Addition, Addition, Equivalent, Equivalent, E...",4
9,Committee on Education and the Workforce of Ho...,Committee on Education and the Workforce of Ho...,[],[Committee on Education and Labor of House of ...,[Addition],[Committee on Education and the Workforce of H...,"[, Committee on Education and the Workforce of...",[Committee on Education and Labor of House of ...,"[Addition, Equivalent]",1


In [31]:
output = result[['paragraph_1', 'paragraph_2', 'span_1_ext', 'span_2_ext', 'Label_ext']].explode(['span_1_ext', 'span_2_ext', 'Label_ext'])

In [29]:
!mkdir rulesm_raw

mkdir: cannot create directory ‘rulesm_raw’: File exists


In [32]:
output.to_csv('rulesm_raw/rulesm_en_with_equivalents.csv', index=False)

In [33]:
output.reset_index(drop=True)

,paragraph_1,paragraph_2,span_1_ext,span_2_ext,Label_ext
0,(A) individual or individual and spouse engage...,(A) individual or individual and spouse engage...,"3,237,000","10,000,000",Contradiction
1,(A) individual or individual and spouse engage...,(A) individual or individual and spouse engage...,(A) individual or individual and spouse engage...,(A) individual or individual and spouse engage...,Equivalent
2,(A) individual or individual and spouse engage...,(A) individual or individual and spouse engage...,and not less than 50 percent of whose aggregat...,and not less than 50 percent of whose aggregat...,Equivalent
3,"(A) subject to subparagraph (B), means a perso...","(A) subject to subparagraph (B), means a perso...",or operating real property or activities incid...,single asset real estate,Contradiction
4,"(A) subject to subparagraph (B), means a perso...","(A) subject to subparagraph (B), means a perso...",for a case in which the United States trustee has,,Addition
...,...,...,...,...,...
144,‘‘(1) the Committee on District of Columbia of...,‘‘(1) the Committee on District of Columbia of...,(1) the Committee on District of Columbia of t...,(1) the Committee on District of Columbia of t...,Equivalent
145,‘‘(1) the Committee on District of Columbia of...,‘‘(1) the Committee on District of Columbia of...,] of the House of Representatives,] of the House of Representatives,Equivalent
146,‘‘(3) Payment of expenses for the procurement ...,‘‘(3) Payment of expenses for the procurement ...,President-elect or Vice-President-elect,apparent successful candidate,Contradiction
147,‘‘(3) Payment of expenses for the procurement ...,‘‘(3) Payment of expenses for the procurement ...,(3) Payment of expenses for the procurement of...,(3) Payment of expenses for the procurement of...,Equivalent
